In [2]:
import os
import random
import shutil
from PIL import Image
import torchvision.transforms as T
from tqdm import tqdm

# ==========================================
# CONFIGURARE CĂI ȘI TARGETURI
# ==========================================
# De unde luăm pozele originale
INPUT_DIR = "../datasets/aptos_reorganized"

# Noul folder unde va sta tot dataset-ul final
OUTPUT_DIR = "../datasets/aptos_augmented_balanced"

TARGETS = {
    'train': 1500,
    'val': 200,
    'test': 250
}

# ==========================================
# PACHETE DE AUGMENTĂRI FIZICE
# ==========================================
# Pentru TRAIN (păstrăm variațiile de lumină)
aug_train = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.RandomRotation(degrees=30),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
    T.RandomAffine(degrees=0, translate=None, scale=(0.9, 1.1))
])

# Pentru VAL și TEST (DOAR flip și rotație, conform cerinței)
aug_eval = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.RandomRotation(degrees=30)
])

def delete_folders_before_augmentation():
    """Șterge folderele de Train, Val și Test din destinație pentru a începe cu un spațiu curat."""
    for split in ['train', 'val', 'test']:
        dst = os.path.join(OUTPUT_DIR, split)
        if os.path.exists(dst):
            print(f"🧹 Ștergem folderul '{split.upper()}' existent pentru a începe curat...")
            shutil.rmtree(dst)

def balance_split(split_name, target_count, transform_pipeline):
    """Echilibrează un singur folder (train, val sau test) la targetul specificat."""
    src_split = os.path.join(INPUT_DIR, split_name)
    dst_split = os.path.join(OUTPUT_DIR, split_name)
    
    if not os.path.exists(src_split):
        print(f"⚠️ Eroare: Folderul sursă '{split_name}' nu există!")
        return
        
    clase = ['0', '1', '2', '3', '4']
    print(f"\n⚖️ Începem echilibrarea setului de {split_name.upper()} (Target: {target_count})...")
    
    for c in clase:
        src_class = os.path.join(src_split, c)
        dst_class = os.path.join(dst_split, c)
        
        os.makedirs(dst_class, exist_ok=True)
        
        if not os.path.exists(src_class):
            continue
            
        # 1. Lista imaginilor originale
        images = [f for f in os.listdir(src_class) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        original_count = len(images)
        
        # 2. Copiem imaginile originale
        print(f"👉 Clasa {c}: Copiem {original_count} imagini originale...")
        for img_name in images:
            shutil.copy2(os.path.join(src_class, img_name), os.path.join(dst_class, img_name))
            
        # 3. Verificăm cât ne lipsește
        needed = target_count - original_count
        
        if needed <= 0:
            print(f"✅ Clasa {c} are deja destule imagini.")
            continue
            
        # 4. Generăm restul
        print(f"⏳ Clasa {c}: Generăm {needed} augmentări...")
        for i in tqdm(range(needed), desc=f"Augmentare {split_name}/Clasa {c}", leave=False):
            img_name = random.choice(images)
            src_img_path = os.path.join(src_class, img_name)
            
            img = Image.open(src_img_path).convert('RGB')
            img_aug = transform_pipeline(img)
            
            nume_baza, ext = os.path.splitext(img_name)
            nume_nou = f"{nume_baza}_aug_{split_name}_{i}{ext}"
            
            img_aug.save(os.path.join(dst_class, nume_nou))

def main():
    print(f"🚀 START: Creare Dataset Complet Echilibrat\n")
    
    # Ștergem folderele existente
    delete_folders_before_augmentation()
    
    # Procesăm fiecare folder în parte cu targetul lui și augmentările specifice
    balance_split('train', TARGETS['train'], aug_train)
    balance_split('val', TARGETS['val'], aug_eval)
    balance_split('test', TARGETS['test'], aug_eval)
    
    print(f"\n✅ OPERAȚIUNE ÎNCHEIATĂ! Toate folderele au fost echilibrate în '{OUTPUT_DIR}'.")

if __name__ == '__main__':
    main()

🚀 START: Creare Dataset Complet Echilibrat

🧹 Ștergem folderul 'TRAIN' existent pentru a începe curat...
🧹 Ștergem folderul 'VAL' existent pentru a începe curat...
🧹 Ștergem folderul 'TEST' existent pentru a începe curat...

⚖️ Începem echilibrarea setului de TRAIN (Target: 1500)...
👉 Clasa 0: Copiem 1434 imagini originale...
⏳ Clasa 0: Generăm 66 augmentări...


👉 Clasa 1: Copiem 300 imagini originale...
⏳ Clasa 1: Generăm 1200 augmentări...


👉 Clasa 2: Copiem 808 imagini originale...
⏳ Clasa 2: Generăm 692 augmentări...


👉 Clasa 3: Copiem 154 imagini originale...
⏳ Clasa 3: Generăm 1346 augmentări...


👉 Clasa 4: Copiem 234 imagini originale...
⏳ Clasa 4: Generăm 1266 augmentări...


...

👉 Clasa 1: Copiem 30 imagini originale...
⏳ Clasa 1: Generăm 220 augmentări...


👉 Clasa 2: Copiem 87 imagini originale...
⏳ Clasa 2: Generăm 163 augmentări...


👉 Clasa 3: Copiem 17 imagini originale...
⏳ Clasa 3: Generăm 233 augmentări...


👉 Clasa 4: Copiem 33 imagini originale...
⏳ Clasa 4: Generăm 217 augmentări...



✅ OPERAȚIUNE ÎNCHEIATĂ! Toate folderele au fost echilibrate în '../datasets/aptos_augmented_balanced'.
